## Import packages and configure paths

The processed Parquet dataset was generated using PySpark in `02_Preprocessing.ipynb`. This notebook uses **batched PyArrow reads** to reduce peak memory on an 8 GB laptop, followed by full-data local LightGBM training. No Spark JVM is needed during model fitting.

In [1]:
# %pip install lightgbm scikit-learn joblib pyarrow

In [2]:
from pathlib import Path
import time
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import joblib
import pyarrow.dataset as ds
from datetime import datetime

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_log_error
)

from config import SPARK_PARQUET_DIR, FIGURE_DIR, TABLE_DIR, MODEL_DIR

In [3]:
# PySpark writes a DIRECTORY with multiple Parquet part-files.
# pandas + PyArrow can read the directory directly; no DuckDB or SparkSession
# is needed for local LightGBM training.
data_file = SPARK_PARQUET_DIR
RANDOM_STATE = 42

print("LightGBM version:", lgb.__version__)
print("PyArrow-backed Parquet directory exists:", data_file.is_dir())
if not data_file.is_dir():
    raise FileNotFoundError(
        f"Spark Parquet directory not found: {data_file}. "
        "Run 02_Preprocessing.ipynb first."
    )


LightGBM version: 4.7.0
PyArrow-backed Parquet directory exists: True


## Define modeling features

In [4]:
# Building and calendar features
base_features = [
    "building_id",
    "site_id",
    "primary_use",
    "square_feet",
    "hour",
    "day_of_week",
    "month"
]

# Weather-related features
weather_features = [
    "air_temperature",
    "dew_temperature",
    "wind_speed"
]

# Complete predictor set
all_features = base_features + weather_features

categorical_features = [
    "building_id",
    "site_id",
    "primary_use"
]

target = "meter_reading"

print("Predictors:", all_features)
print("Target:", target)

Predictors: ['building_id', 'site_id', 'primary_use', 'square_feet', 'hour', 'day_of_week', 'month', 'air_temperature', 'dew_temperature', 'wind_speed']
Target: meter_reading


## Create chronological datasets — full training period

Read **all** eligible January–September 2016 observations and October–November validation observations from the Spark-generated Parquet directory. Rather than materializing millions of rows into a large temporary 64-bit pandas DataFrame, a PyArrow scanner yields batches, which are converted to compact types before concatenation. December is not loaded until final evaluation. No sampling is used.

In [ ]:
columns = all_features + [target]

# Dataset is a Spark-generated Parquet DIRECTORY of part-files.
# Reading it as one pandas DataFrame can exhaust RAM before type conversion.
arrow_data = ds.dataset(str(data_file), format="parquet")

# Primary-use categories are small and known from building metadata.
# Using the SAME categories for each batch prevents pd.concat from
# converting the category column back to large Python strings.
from config import RAW_DIR
building_meta = pd.read_csv(
    RAW_DIR / "building_metadata.csv", usecols=["primary_use"]
)
use_categories = sorted(building_meta["primary_use"].dropna().unique())
del building_meta

numeric_features = [
    "square_feet", "hour", "day_of_week", "month",
    "air_temperature", "dew_temperature", "wind_speed"
]

def read_period_in_batches(start=None, end=None, batch_size=65536):
    """Load an entire date interval with low intermediate memory overhead."""
    condition = None
    if start is not None:
        condition = ds.field("timestamp") >= datetime.fromisoformat(start)
    if end is not None:
        upper = ds.field("timestamp") < datetime.fromisoformat(end)
        condition = upper if condition is None else condition & upper

    scanner = arrow_data.scanner(
        columns=columns,
        filter=condition,
        batch_size=batch_size,
        use_threads=False
    )
    chunks = []
    count = 0
    for batch in scanner.to_batches():
        part = batch.to_pandas()
        # Convert EACH batch before storing it; do not retain raw strings
        # or double-precision numerical arrays for the full dataset.
        part["primary_use"] = pd.Categorical(
            part["primary_use"], categories=use_categories
        )
        part["building_id"] = part["building_id"].astype("int16")
        part["site_id"] = part["site_id"].astype("int8")
        for name in numeric_features + [target]:
            part[name] = part[name].astype("float32")
        chunks.append(part)
        count += len(part)
        if len(chunks) % 40 == 0:
            print(f"  Loaded {count:,} records...", flush=True)
    if not chunks:
        raise ValueError(f"No records found between {start} and {end}.")
    result = pd.concat(chunks, ignore_index=True, copy=False)
    del chunks
    gc.collect()
    print(
        f"Finished {count:,} records; DataFrame memory: "
        f"{result.memory_usage(deep=True).sum()/1024**2:.1f} MiB"
    )
    return result

print("Loading full January–September training period...")
train_df = read_period_in_batches(end="2016-10-01")
print("Loading October–November validation period...")
val_df = read_period_in_batches(start="2016-10-01", end="2016-12-01")

print("Full training records:", len(train_df))
print("Validation records:", len(val_df))
assert len(train_df) > 0 and len(val_df) > 0


## Prepare data types

In [ ]:
# The loader has already converted numerical features/target to float32.
# Match categorical levels to the TRAINING period only, for validation and
# eventual December test data. Removes unused metadata categories.
category_levels = {}
for feature in categorical_features:
    train_df[feature] = train_df[feature].astype("category")
    train_df[feature] = train_df[feature].cat.remove_unused_categories()
    category_levels[feature] = train_df[feature].cat.categories
    val_df[feature] = pd.Categorical(
        val_df[feature], categories=category_levels[feature]
    )

for frame in (train_df, val_df):
    assert frame[target].notna().all()
    assert (frame[target] >= 0).all()

print("Full training records:", len(train_df))
print("Training memory (MiB):", round(train_df.memory_usage(deep=True).sum()/1024**2, 1))
print("Validation memory (MiB):", round(val_df.memory_usage(deep=True).sum()/1024**2, 1))


## Define evaluation metrics

In [ ]:
def evaluate_model(y_true, y_pred):
    y_pred = np.maximum(y_pred, 0)

    mae = mean_absolute_error(
        y_true, y_pred
    )

    rmsle = np.sqrt(
        mean_squared_log_error(
            y_true, y_pred
        )
    )

    return {
        "MAE": round(mae, 4),
        "RMSLE": round(rmsle, 4)
    }

# Train and evaluate the models

## Historical baseline prediction

In [ ]:
# Historical median for each building and hour
baseline_table = (
    train_df
    .groupby(
        ["building_id", "hour"],
        observed=True
    )[target]
    .median()
    .rename("baseline_prediction")
    .reset_index()
)

# Fallback for an unseen building-hour combination
building_median = (
    train_df
    .groupby("building_id", observed=True)[target]
    .median()
)

global_median = train_df[target].median()

def baseline_predict(df):
    result = df[
        ["building_id", "hour"]
    ].merge(
        baseline_table,
        on=["building_id", "hour"],
        how="left",
        sort=False
    )

    predictions = result["baseline_prediction"]

    predictions = predictions.fillna(
        df["building_id"].map(building_median).reset_index(drop=True)
    )

    return predictions.fillna(global_median).to_numpy()

baseline_val_pred = baseline_predict(val_df)

baseline_results = evaluate_model(
    val_df[target],
    baseline_val_pred
)

print("Historical baseline:", baseline_results)

## Train LightGBM without weather variables

In [ ]:
# Model 1: LightGBM WITHOUT weather features, trained on the full period
# Select only the required columns so we do not keep unnecessary feature copies.
X_train_base = train_df[base_features]
X_val_base = val_df[base_features]
y_train = np.log1p(train_df[target].to_numpy(dtype="float32"))
y_val = val_df[target].to_numpy(dtype="float32")

model_params = {
    "objective": "regression",
    "n_estimators": 200,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": -1,
    "max_bin": 127,
    "n_jobs": 2,
    "verbosity": -1,
    "random_state": RANDOM_STATE
}

model_base = lgb.LGBMRegressor(**model_params)
start = time.perf_counter()
model_base.fit(
    X_train_base, y_train,
    categorical_feature=categorical_features
)
base_training_time = time.perf_counter() - start
base_log_pred = model_base.predict(X_val_base)
base_pred = np.maximum(np.expm1(base_log_pred), 0)
base_results = evaluate_model(y_val, base_pred)

# Free the first model's feature matrices before constructing the second.
del X_train_base, X_val_base, base_log_pred
gc.collect()

print("Without weather:", base_results)
print("Training time:", round(base_training_time, 2), "seconds")


## Train LightGBM with weather variables

In [ ]:
# Model 2: the SAME LightGBM configuration, now WITH weather features
X_train_all = train_df[all_features]
X_val_all = val_df[all_features]
model_weather = lgb.LGBMRegressor(**model_params)

start = time.perf_counter()
model_weather.fit(
    X_train_all, y_train,
    categorical_feature=categorical_features
)
weather_training_time = time.perf_counter() - start
weather_log_pred = model_weather.predict(X_val_all)
weather_pred = np.maximum(np.expm1(weather_log_pred), 0)
weather_results = evaluate_model(y_val, weather_pred)

del X_train_all, X_val_all, weather_log_pred
gc.collect()

print("With weather:", weather_results)
print("Training time:", round(weather_training_time, 2), "seconds")


## Compare validation results

In [ ]:
comparison = pd.DataFrame([
    {
        "Model": "Historical Baseline",
        **baseline_results,
        "Training Time (s)": 0.0
    },
    {
        "Model": "LightGBM without Weather",
        **base_results,
        "Training Time (s)": round(base_training_time, 2)
    },
    {
        "Model": "LightGBM with Weather",
        **weather_results,
        "Training Time (s)": round(weather_training_time, 2)
    }
])

display(comparison)

comparison.to_csv(
    TABLE_DIR / "validation_results.csv",
    index=False
)

# Final evaluation using December 2016

## Evaluate on December 2016 (untouched test set)

Load the December records from the Spark-generated Parquet directory only after selecting the LightGBM feature configuration using October–November validation RMSLE.

In [ ]:
# Choose the feature configuration using VALIDATION results only
if weather_results["RMSLE"] < base_results["RMSLE"]:
    final_model = model_weather
    final_features = all_features
    selected_name = "LightGBM with Weather"
else:
    final_model = model_base
    final_features = base_features
    selected_name = "LightGBM without Weather"

print("Selected configuration:", selected_name)

# Load December only now, in compact batches.
print("Loading December test period...")
test_df = read_period_in_batches(start="2016-12-01")
for feature in categorical_features:
    test_df[feature] = pd.Categorical(
        test_df[feature], categories=category_levels[feature]
    )

assert test_df[target].notna().all()
assert (test_df[target] >= 0).all()
print("December test records:", len(test_df))
print("December memory (MiB):", round(test_df.memory_usage(deep=True).sum()/1024**2, 1))

test_log_pred = final_model.predict(test_df[final_features])
test_pred = np.maximum(np.expm1(test_log_pred), 0)
del test_log_pred
gc.collect()

baseline_test_pred = baseline_predict(test_df)
test_results = evaluate_model(test_df[target], test_pred)
baseline_test_results = evaluate_model(test_df[target], baseline_test_pred)

final_results = pd.DataFrame([
    {"Model": "Historical Baseline", **baseline_test_results},
    {"Model": selected_name, **test_results}
])
display(final_results)
final_results.to_csv(TABLE_DIR / "final_test_results.csv", index=False)


## Plot actual versus predicted electricity consumption

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)

sample_size = min(10000, len(test_df))

indices = rng.choice(
    len(test_df),
    size=sample_size,
    replace=False
)

actual = test_df[target].to_numpy()[indices]
predicted = test_pred[indices]

plt.figure(figsize=(7, 6))

plt.scatter(
    actual,
    predicted,
    alpha=0.15,
    s=8
)

limit = np.percentile(
    np.concatenate([actual, predicted]), 99
)

plt.plot(
    [0, limit],
    [0, limit],
    linestyle="--",
    color="black",
    linewidth=1.5,
    label="Perfect prediction"
)

plt.xlim(0, limit)
plt.ylim(0, limit)

plt.xlabel("Actual Electricity Meter Reading")
plt.ylabel("Predicted Electricity Meter Reading")
plt.title("Actual vs Predicted Electricity Consumption")

plt.legend()
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "actual_vs_predicted_electricity_prediction.png",
    bbox_inches="tight"
)

plt.show()

## Save the model

In [ ]:
model_file = MODEL_DIR / "electricity_lightgbm_full.joblib"

joblib.dump(
    {
        "model": final_model,
        "features": final_features,
        "model_name": selected_name,
        "data_source": "PySpark-generated Parquet (electricity_data_spark.parquet)",
        "parameters": model_params,
        "training_period": "2016-01 to 2016-09",
        "validation_period": "2016-10 to 2016-11",
        "test_period": "2016-12"
    },
    model_file
)

print("Model saved to:", model_file)